# 2. Training

Trains every model needed for the evaluation and saves it (scaler + model) to `models/`. Every model is listed in `models/manifest.csv`, with the experiment, split, feature set, pooling, model, hyperparameters and training time. The splits are saved in `models/splits.csv` and the feature definitions in `models/feature_specs.json`, so `03_evaluation.ipynb` evaluates on exactly the same data.

| Experiment | Split | What is trained |
|---|---|---|
| `feature_study` | recording-level (seed 42) | tuned SVM on feature variants: each feature group left out, 13/20/40 MFCCs, MFCC 0 dropped, window length 1024/2048/4096 |
| `leakage` | random excerpt-level and recording-level (seed 42) | default models, combined features, mean + std pooling |
| `comparison` | recording-level (seed 42) | 4 feature sets × 2 poolings × 4 models, default and tuned |
| `repeated` | 5 further recording-level splits (seeds 0-4) | tuned models of the best configuration |
| `learning_curve` | recording-level (seed 42) | tuned models of the best configuration on 10% / 25% / 50% of the training data |
| `illustration` | recording-level (seed 42) | tuned models on the first 2 principal components (to draw decision boundaries) |
| `official` | recording-level (seed 42) | tuned SVM of the best configuration with probability estimates (for the official test set) |

All choices (feature settings, hyperparameters, best feature set and pooling) are made on the **validation** split only.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.config import (TRAIN_DATA_DIR, CACHE_DIR, MODELS_DIR, SEED, REPEAT_SEEDS, FEATURE_SETS, FEATURE_SET_GROUPS, POOLINGS,
                        HOP_LENGTH, DEFAULT_FEATURE_PARAMS, feature_spec)
from src.data import load_dataset, split_dataset, group_split_dataset
from src.features import build_features
from src.models import MODEL_NAMES
from src.experiments import PARAM_GRIDS, MODEL_SLUGS, scaled_split, timed_fit, tune_model, splits_table, save_model

MODELS_DIR.mkdir(parents=True, exist_ok=True)
tuning_dir = MODELS_DIR / "tuning"
tuning_dir.mkdir(exist_ok=True)

df = load_dataset(TRAIN_DATA_DIR)
print("Excerpts:", len(df))

Excerpts: 6705


## Splits

The random split is the one used in the first experiments: excerpts of the same recording can end up in both train and test. All other splits are recording-level: every recording is in exactly one of train / validation / test (70/15/15, stratified by instrument).

In [2]:
splits = {
    f"random_seed{SEED}": split_dataset(df, seed=SEED),
    f"grouped_seed{SEED}": group_split_dataset(df, seed=SEED),
    **{f"grouped_seed{seed}": group_split_dataset(df, seed=seed) for seed in REPEAT_SEEDS}
}

for split_name, (train_df, val_df, test_df) in splits.items():
    test_in_train = test_df["recording_id"].isin(train_df["recording_id"]).mean()
    print(f"{split_name:18s} train {len(train_df):5d}  val {len(val_df):5d}  test {len(test_df):5d}  "
          f"test excerpts whose recording is in train: {test_in_train:.1%}")
    if split_name.startswith("grouped"):
        assert test_in_train == 0 and not val_df["recording_id"].isin(train_df["recording_id"]).any()

splits_table(df, splits).to_csv(MODELS_DIR / "splits.csv", index=False)

random_seed42      train  4693  val  1006  test  1006  test excerpts whose recording is in train: 93.0%
grouped_seed42     train  4692  val  1006  test  1007  test excerpts whose recording is in train: 0.0%
grouped_seed0      train  4692  val  1007  test  1006  test excerpts whose recording is in train: 0.0%
grouped_seed1      train  4690  val  1007  test  1008  test excerpts whose recording is in train: 0.0%
grouped_seed2      train  4691  val  1006  test  1008  test excerpts whose recording is in train: 0.0%
grouped_seed3      train  4690  val  1007  test  1008  test excerpts whose recording is in train: 0.0%
grouped_seed4      train  4692  val  1006  test  1007  test excerpts whose recording is in train: 0.0%


In [3]:
manifest = []

def split_data(split_name, x):
    train_df, val_df, test_df = splits[split_name]
    x_train, x_val, _, scaler = scaled_split(x, train_df, val_df, test_df)
    return x_train, train_df["label"].values, x_val, val_df["label"].values, scaler

def manifest_params(experiment, feature_set, pooling, model_name, variant="tuned"):
    return next(json.loads(row["params"]) for row in manifest
                if row["experiment"] == experiment and row["feature_set"] == feature_set and row["pooling"] == pooling
                and row["model"] == model_name and row["variant"] == variant)

## Experiment `feature_study`: choosing the features and their parameters

Starting from the extended feature set with the default frame settings (13 MFCCs, MFCC 0 kept, 2048-sample window), one choice is changed at a time:
- **Ablation:** each feature group is left out in turn. The spectral shape descriptors (rolloff, flatness, bandwidth) form one group, and so do the original low-level features (centroid, ZCR).
- **Number of MFCCs:** 20 or 40 instead of 13.
- **MFCC 0:** dropped.
- **Window length:** 1024 or 4096 samples instead of 2048. The hop stays at 512 samples, so only the frequency/time resolution changes, not the frame rate.

Each variant is evaluated with a tuned SVM (the best model of the first experiments, retuned for every variant because the dimensionality changes) and mean + std pooling. The combined set of the first experiments is included as a reference (named as `study_combined`).

**Selection rule (validation macro F1):**
- a feature group is removed only if leaving it out *improves* the score;
- the number of MFCCs, MFCC 0 and the window length are each set to their best value.

The resulting settings define the `extended` feature set used in all later experiments.

In [4]:
ABLATIONS = {
    "delta_mfcc": ["delta_mfcc"],
    "spectral_contrast": ["spectral_contrast"],
    "spectral_shape": ["spectral_rolloff", "spectral_flatness", "spectral_bandwidth"],
    "centroid_zcr": ["spectral_centroid", "zcr"]
}

def without(groups):
    return [group for group in FEATURE_SET_GROUPS["extended"] if group not in groups]

study_specs = {
    "study_combined": feature_spec("combined"),
    "study_extended": feature_spec("extended"),
    **{f"study_no_{name}": feature_spec("extended", groups=without(groups)) for name, groups in ABLATIONS.items()},
    "study_n_mfcc_20": feature_spec("extended", n_mfcc=20),
    "study_n_mfcc_40": feature_spec("extended", n_mfcc=40),
    "study_drop_c0": feature_spec("extended", drop_c0=True),
    "study_n_fft_1024": feature_spec("extended", n_fft=1024),
    "study_n_fft_4096": feature_spec("extended", n_fft=4096)
}

# One cached extraction per window length; everything else is a column selection
study_features = build_features(df, study_specs, ["mean_std"], CACHE_DIR, hop_length=HOP_LENGTH)["mean_std"]

split_name = f"grouped_seed{SEED}"
study_val_f1 = {}
for name, x in study_features.items():
    x_train, y_train, x_val, y_val, scaler = split_data(split_name, x)
    model, params, tuning_table = tune_model("SVM", x_train, y_train, x_val, y_val, seed=SEED)
    manifest.append(save_model(MODELS_DIR, "feature_study", split_name, name, "mean_std", "SVM", "tuned", model, scaler, params, dimensions=x.shape[1]))
    study_val_f1[name] = tuning_table["Val macro F1"].max()
    print(f"{name:28s} {x.shape[1]:4d} dims   val macro F1 {study_val_f1[name]:.4f}")

study_combined                 30 dims   val macro F1 0.5148
study_extended                 76 dims   val macro F1 0.5510
study_no_delta_mfcc            50 dims   val macro F1 0.5684
study_no_spectral_contrast     62 dims   val macro F1 0.5044
study_no_spectral_shape        70 dims   val macro F1 0.5499
study_no_centroid_zcr          72 dims   val macro F1 0.5505
study_n_mfcc_20               104 dims   val macro F1 0.5521
study_n_mfcc_40               184 dims   val macro F1 0.5603
study_drop_c0                  72 dims   val macro F1 0.5392
study_n_fft_1024               76 dims   val macro F1 0.5707
study_n_fft_4096               76 dims   val macro F1 0.5669


In [5]:
baseline_f1 = study_val_f1["study_extended"]

removed_groups = [group for name, groups in ABLATIONS.items() if study_val_f1[f"study_no_{name}"] > baseline_f1 for group in groups]
n_mfcc_scores = {13: baseline_f1, 20: study_val_f1["study_n_mfcc_20"], 40: study_val_f1["study_n_mfcc_40"]}
n_fft_scores = {1024: study_val_f1["study_n_fft_1024"], 2048: baseline_f1, 4096: study_val_f1["study_n_fft_4096"]}

extended_spec = feature_spec(
    "extended",
    groups=without(removed_groups),
    n_mfcc=max(n_mfcc_scores, key=n_mfcc_scores.get),
    n_fft=max(n_fft_scores, key=n_fft_scores.get),
    drop_c0=bool(study_val_f1["study_drop_c0"] > baseline_f1)
)

print("Removed groups:", removed_groups or "none")
extended_spec

Removed groups: ['delta_mfcc']


{'groups': ['mfcc',
  'spectral_centroid',
  'zcr',
  'spectral_contrast',
  'spectral_rolloff',
  'spectral_flatness',
  'spectral_bandwidth'],
 'n_mfcc': 40,
 'n_fft': 1024,
 'drop_c0': False}

## Feature sets for the remaining experiments

The three feature sets of the first experiments keep their original settings, so they stay comparable. `extended` uses the settings chosen above.

In [6]:
specs = {**{feature_set: feature_spec(feature_set) for feature_set in FEATURE_SETS if feature_set != "extended"}, "extended": extended_spec}

with open(MODELS_DIR / "feature_specs.json", "w") as f:
    json.dump({**study_specs, **specs}, f, indent=2)

features = build_features(df, specs, POOLINGS, CACHE_DIR, hop_length=HOP_LENGTH)

for pooling in POOLINGS:
    print(pooling, {name: x.shape[1] for name, x in features[pooling].items()})

mean {'mfcc': 13, 'low_level': 2, 'combined': 15, 'extended': 52}
mean_std {'mfcc': 26, 'low_level': 4, 'combined': 30, 'extended': 104}


## Experiment `leakage`: random vs. recording-level split

Same features, same default models; only the split differs.

In [7]:
for split_name in [f"random_seed{SEED}", f"grouped_seed{SEED}"]:
    x_train, y_train, x_val, y_val, scaler = split_data(split_name, features["mean_std"]["combined"])
    for model_name in MODEL_NAMES:
        model, seconds = timed_fit(model_name, x_train, y_train, x_val, y_val, seed=SEED)
        manifest.append(save_model(MODELS_DIR, "leakage", split_name, "combined", "mean_std", model_name, "default", model, scaler, fit_seconds=seconds))

## Experiment `comparison`: feature set × pooling × model, default and tuned

Each model is trained with the default settings of the first experiments (kNN: scikit-learn's default, k = 5), and tuned by grid search on the validation split (selected by macro F1). The grid search results are saved in `models/tuning/`.

In [8]:
PARAM_GRIDS

{'Logistic Regression': {'C': [0.01, 0.1, 1.0, 10.0, 100.0]},
 'kNN': {'n_neighbors': [1, 3, 5, 11, 21, 41],
  'weights': ['uniform', 'distance'],
  'p': [1, 2]},
 'SVM': {'C': [0.1, 1.0, 10.0, 100.0], 'gamma': ['scale', 0.003, 0.01, 0.1]},
 'MLP': {'hidden_dims': [(64, 32), (256, 128)],
  'dropout': [0.0, 0.3],
  'weight_decay': [0.0, 0.0001],
  'epochs': [300],
  'patience': [20]}}

In [9]:
split_name = f"grouped_seed{SEED}"
best_val_f1 = {}

for pooling in POOLINGS:
    for feature_set in FEATURE_SETS:
        x_train, y_train, x_val, y_val, scaler = split_data(split_name, features[pooling][feature_set])
        
        for model_name in MODEL_NAMES:
            model, seconds = timed_fit(model_name, x_train, y_train, x_val, y_val, seed=SEED)
            manifest.append(save_model(MODELS_DIR, "comparison", split_name, feature_set, pooling, model_name, "default", model, scaler, fit_seconds=seconds))
            
            model, params, tuning_table = tune_model(model_name, x_train, y_train, x_val, y_val, seed=SEED)
            manifest.append(save_model(MODELS_DIR, "comparison", split_name, feature_set, pooling, model_name, "tuned", model, scaler, params))
            tuning_table.to_csv(tuning_dir / f"{feature_set}_{pooling}_{MODEL_SLUGS[model_name]}.csv", index=False)
            best_val_f1[(feature_set, pooling, model_name)] = tuning_table["Val macro F1"].max()
        
        print("done:", feature_set, pooling)

done: mfcc mean
done: low_level mean
done: combined mean
done: extended mean
done: mfcc mean_std
done: low_level mean_std
done: combined mean_std
done: extended mean_std


In [10]:
# Best feature set + pooling: the one with the best tuned model on validation
best_feature_set, best_pooling, best_model_name = max(best_val_f1, key=best_val_f1.get)
best_config = {"feature_set": best_feature_set, "pooling": best_pooling, "model": best_model_name, "val_macro_f1": best_val_f1[(best_feature_set, best_pooling, best_model_name)]}

with open(MODELS_DIR / "best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)

x_best = features[best_pooling][best_feature_set]
best_config

{'feature_set': 'extended',
 'pooling': 'mean_std',
 'model': 'MLP',
 'val_macro_f1': np.float64(0.5830467734328028)}

## Experiment `repeated`: 5 more recording-level splits

The best configuration is tuned and trained again on 5 different splits (the MLP is seeded with the split seed), to measure how much the results vary.

In [11]:
for seed in REPEAT_SEEDS:
    split_name = f"grouped_seed{seed}"
    x_train, y_train, x_val, y_val, scaler = split_data(split_name, x_best)
    
    for model_name in MODEL_NAMES:
        model, params, _ = tune_model(model_name, x_train, y_train, x_val, y_val, seed=seed)
        manifest.append(save_model(MODELS_DIR, "repeated", split_name, best_feature_set, best_pooling, model_name, "tuned", model, scaler, params))
    
    print("done: seed", seed)

done: seed 0
done: seed 1
done: seed 2
done: seed 3
done: seed 4


## Experiment `learning_curve`: how much data does each model need?

The tuned settings of the best configuration, trained on a stratified random 10% / 25% / 50% of the training split (100% is the `comparison` model). The scaler is fit on the subset only.

In [12]:
split_name = f"grouped_seed{SEED}"
train_df, val_df, test_df = splits[split_name]

for fraction in [0.1, 0.25, 0.5]:
    subset_df, _ = train_test_split(train_df, train_size=fraction, stratify=train_df["label"], random_state=SEED)
    x_train, x_val, _, scaler = scaled_split(x_best, subset_df, val_df, test_df)
    
    for model_name in MODEL_NAMES:
        params = manifest_params("comparison", best_feature_set, best_pooling, model_name)
        model, seconds = timed_fit(model_name, x_train, subset_df["label"].values, x_val, val_df["label"].values, params=params, seed=SEED)
        manifest.append(save_model(MODELS_DIR, "learning_curve", split_name, best_feature_set, best_pooling, model_name, f"tuned_frac{fraction}",
                                   model, scaler, params, fit_seconds=seconds, train_fraction=fraction, train_size=len(subset_df)))

## Experiment `illustration`: decision boundaries in 2D

To *see* how the models differ, each model is trained (with its tuned settings) on only the first two principal components of the best feature set. The scores of these 2D models are much lower than with all features; they are only meant for drawing the decision regions in `03_evaluation.ipynb`.

In [13]:
split_name = f"grouped_seed{SEED}"
train_df, val_df, test_df = splits[split_name]
projection = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=SEED)).fit(x_best[train_df.index])
x_train_2d, x_val_2d = projection.transform(x_best[train_df.index]), projection.transform(x_best[val_df.index])

for model_name in MODEL_NAMES:
    params = manifest_params("comparison", best_feature_set, best_pooling, model_name)
    model, seconds = timed_fit(model_name, x_train_2d, train_df["label"].values, x_val_2d, val_df["label"].values, params=params, seed=SEED)
    manifest.append(save_model(MODELS_DIR, "illustration", split_name, best_feature_set, best_pooling, model_name, "tuned_pca2", model, projection, params, fit_seconds=seconds))

## Experiment `official`: SVM with probability estimates

The official test set is evaluated by averaging class probabilities over windows. Logistic Regression, kNN and the MLP give probabilities directly, so their tuned models from `comparison` are used. The SVM is refit with the same tuned settings and Platt scaling.

In [14]:
split_name = f"grouped_seed{SEED}"
x_train, y_train, x_val, y_val, scaler = split_data(split_name, x_best)

svm_params = manifest_params("comparison", best_feature_set, best_pooling, "SVM")
model, seconds = timed_fit("SVM", x_train, y_train, x_val, y_val, params=svm_params, seed=SEED, probability=True)
manifest.append(save_model(MODELS_DIR, "official", split_name, best_feature_set, best_pooling, "SVM", "tuned_proba", model, scaler, svm_params, fit_seconds=seconds))

In [15]:
manifest = pd.DataFrame(manifest)
manifest.to_csv(MODELS_DIR / "manifest.csv", index=False)

manifest.groupby(["experiment", "variant"]).size()

experiment      variant       
comparison      default           32
                tuned             32
feature_study   tuned             11
illustration    tuned_pca2         4
leakage         default            8
learning_curve  tuned_frac0.1      4
                tuned_frac0.25     4
                tuned_frac0.5      4
official        tuned_proba        1
repeated        tuned             20
dtype: int64